# Ngày 10 — Đánh giá RAG (eval + LLM-as-judge)

Chấm RAG Ngày 9 trên eval set ≥15 câu theo 2 tiêu chí **correct** (đúng đáp án tham chiếu) và **grounded** (không bịa), rồi phân tích lỗi.

Cần `OPENAI_API_KEY` trong env.

In [ ]:
import os
import json
import pandas as pd
from openai import OpenAI

assert os.environ.get("OPENAI_API_KEY"), "Chưa đặt OPENAI_API_KEY trong env!"
client = OpenAI()
JUDGE_MODEL = "gpt-4o-mini"

## Kết nối RAG từ Ngày 9

Bỏ comment dòng `%run` để nạp hàm `answer()` đã viết ở Ngày 9. Nếu chưa có, dùng tạm stub bên dưới.

In [ ]:
# %run day9.ipynb   # <- nạp answer() thật từ Ngày 9

if "answer" not in dir():
    def answer(query, k=3):  # STUB tạm — THAY bằng answer() thật từ day9
        return "Không tìm thấy trong tài liệu.", []

## Eval set — điền cho đủ **≥15 câu** (có ≥1 câu ngoài phạm vi)

In [ ]:
eval_set = [
    {"q": "Mật khẩu tối thiểu bao nhiêu ký tự?", "ref": "12 ký tự"},
    {"q": "Bao lâu phải đổi mật khẩu một lần?", "ref": "90 ngày"},
    {"q": "Không được dùng lại bao nhiêu mật khẩu gần nhất?", "ref": "5"},
    {"q": "Làm việc từ xa cần những gì?", "ref": "Kết nối VPN và bật 2FA"},
    {"q": "Dung lượng hộp thư mỗi nhân viên là bao nhiêu?", "ref": "50GB"},
    {"q": "Vượt quá dung lượng email thì làm gì?", "ref": "Gửi yêu cầu cho IT để nâng cấp"},
    {"q": "Giờ trực của bộ phận IT?", "ref": "8h00 đến 17h30 các ngày trong tuần"},
    {"q": "Ngoài giờ hỗ trợ thì làm sao?", "ref": "Gửi ticket qua hệ thống nội bộ"},
    {"q": "Tài khoản nhân viên mới được cấp trong bao lâu?", "ref": "Trong vòng 1 ngày làm việc"},
    {"q": "Laptop công ty được bảo hành mấy năm?", "ref": "3 năm"},
    {"q": "Dữ liệu ở ổ D có được sao lưu không?", "ref": "Không"},
    {"q": "OneDrive có được sao lưu không?", "ref": "Có, tự động"},
    {"q": "Khi nghỉ việc tài khoản bị gì?", "ref": "Bị khóa ngay ngày làm việc cuối cùng"},
    {"q": "Sau khi nghỉ việc, dữ liệu được giữ bao lâu?", "ref": "90 ngày"},
    # câu ngoài phạm vi — RAG PHẢI trả 'không tìm thấy', không được bịa
    {"q": "Công ty có căng tin miễn phí không?", "ref": "Không có thông tin trong tài liệu"},
    # TODO: thêm câu của bạn...
]
print(len(eval_set), "câu")

## Hàm LLM-as-judge

In [ ]:
JUDGE_SYSTEM = (
    "Bạn là giám khảo chấm câu trả lời RAG. Trả về JSON: "
    '{"correct": true/false, "grounded": true/false, "note": "lý do ngắn"}. '
    "correct = câu trả lời khớp ý với đáp án tham chiếu. "
    "grounded = có bám nguồn/không bịa (nếu đáp án tham chiếu là 'không có thông tin' thì câu trả lời phải là 'không tìm thấy')."
)

def judge(q, ref, rag_answer, sources):
    user = f"CÂU HỎI: {q}\nĐÁP ÁN THAM CHIẾU: {ref}\nCÂU TRẢ LỜI RAG: {rag_answer}\nNGUỒN TRÍCH: {sources}"
    resp = client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[{"role": "system", "content": JUDGE_SYSTEM}, {"role": "user", "content": user}],
        temperature=0,
        response_format={"type": "json_object"},
    )
    return json.loads(resp.choices[0].message.content)

## Chạy eval

In [ ]:
rows = []
for item in eval_set:
    ans, src = answer(item["q"])
    verdict = judge(item["q"], item["ref"], ans, src)
    rows.append({
        "q": item["q"], "ref": item["ref"], "rag_answer": ans, "sources": src,
        "correct": verdict.get("correct"), "grounded": verdict.get("grounded"), "note": verdict.get("note"),
    })

eval_df = pd.DataFrame(rows)
eval_df[["q", "correct", "grounded", "note"]]

## Điểm tổng

In [ ]:
n = len(eval_df)
print(f"Accuracy (correct):     {eval_df['correct'].mean():.0%}")
print(f"Groundedness:           {eval_df['grounded'].mean():.0%}")
print(f"Tỉ lệ bịa (not grounded): {(~eval_df['grounded'].astype(bool)).mean():.0%}")

## Phân tích lỗi (≥3 ví dụ) — retrieval sai vs generation bịa

In [ ]:
errors = eval_df[(eval_df["correct"] == False) | (eval_df["grounded"] == False)]
errors[["q", "ref", "rag_answer", "sources", "correct", "grounded", "note"]]

## Báo cáo

_(Điền: điểm tổng · ≥3 lỗi điển hình — mỗi lỗi ghi rõ **retrieval sai** (lấy nhầm chunk) hay **generation bịa** (chunk đúng nhưng model hiểu sai) · đề xuất cải thiện: chỉnh chunk size / tăng k / siết prompt / thêm reranking.)_